# Sultan Hassan Master Pipeline (Captions + Training)
This notebook combines the Caption Helper and the Task-Compliant Training into one smooth pipeline to ensure you get full points on the assignment!
It includes Drive auto-saving, memory optimizations, and correct `metadata.jsonl` generation.

In [ ]:
# Step 0: Mount Google Drive to Auto-Save
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Step 1: Install Dependencies
!pip install -q -U \
  diffusers==0.31.0 \
  transformers==4.46.3 \
  accelerate==1.1.1 \
  peft==0.13.2 \
  datasets==3.1.0 \
  xformers \
  wandb

try:
    import xformers
    USE_XFORMERS = True
except Exception as e:
    USE_XFORMERS = False


In [ ]:
# Step 2: Upload RAW Images Zip (Just images, no captions yet)
import os
from google.colab import files

print("Upload your zip of RAW Sultan Hassan images:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

!rm -rf dataset_raw
!unzip -q -o {zip_name} -d dataset_raw

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp"}

def find_content_dir(root):
    for dirpath, _, filenames in os.walk(root):
        if any(os.path.splitext(f)[1].lower() in IMG_EXTS for f in filenames):
            return dirpath
    return None

src_dir = find_content_dir("dataset_raw")
images = sorted([f for f in os.listdir(src_dir) if os.path.splitext(f)[1].lower() in IMG_EXTS])
print(f"Found {len(images)} images!")


In [ ]:
# Step 3: Preview Thumbnails & Get Filenames
from PIL import Image
import matplotlib.pyplot as plt

cols = 5
rows = (len(images) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten()

for i, fname in enumerate(images):
    img = Image.open(os.path.join(src_dir, fname))
    axes[i].imshow(img)
    axes[i].set_title(fname, fontsize=9)
    axes[i].axis("off")
for j in range(len(images), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Step 4: MANUALLY TYPE YOUR CAPTIONS HERE
CAPTIONS = {
    "1.jpg": "exterior facade, massive plain stone walls, vertical window bays",
    "2.jpg": "monumental entrance portal with intricate muqarnas detailing",
    "3.jpg": "courtyard with four iwans, midday sunlight hitting the stone",
    "4.jpg": "minaret rising against a clear blue sky",
    "5.jpg": "dome exterior with stone cresting along the roofline",
    "6.jpg": "interior ornament showing muqarnas ceiling detail",
    "7.jpg": "hanging glass lamp inside the dark prayer hall",
    "8.jpg": "wide view of the central open courtyard and fountain",
    "9.jpg": "massive stone archway of the main qibla iwan",
    "10.jpg": "exterior view showing the sheer height of the Mamluk architecture",
    "11.jpg": "close up of the bronze entrance doors with geometric patterns",
    "12.jpg": "sunlight casting shadows across the courtyard walls",
    "13.jpg": "interior view looking up at the soaring vaulted ceiling",
    "14.jpg": "marble paneling and mihrab inside the main sanctuary",
    "15.jpg": "silhouette of the mosque at night against the dark sky",
    "16.jpg": "exterior facade captured during sunset with warm lighting",
    "17.jpg": "courtyard fountain pavilion with a wooden dome",
    "18.jpg": "intricate stone carving around the tall arched windows",
    "19.jpg": "hanging lamps suspended from chains in the dim interior",
    "20.jpg": "view from the courtyard looking toward the towering minaret",
    "21.jpg": "close up of the muqarnas stalactite corbeling over the entrance",
    "22.jpg": "massive stone blocks forming the outer fortress-like wall",
    "23.jpg": "interior corridor with arched stone ceiling",
    "24.jpg": "stained glass window filtering colored light into the hall",
    "25.jpg": "view of the mosque complex from a distance",
    "26.jpg": "the large courtyard completely empty in the early morning",
    "27.jpg": "detailed view of the kufic calligraphy carved into the stone",
    "28.jpg": "stone steps leading up to the monumental entrance gates",
    "29.jpg": "the dome and minarets viewed from street level",
    "30.jpg": "sun rays piercing through the windows of the prayer hall",
    "31.jpg": "geometric star patterns carved into the wooden doors",
    "32.jpg": "the ablution fountain centered in the massive stone courtyard",
    "33.jpg": "exterior wall showing the alternating colored stone masonry (ablaq)",
    "34.jpg": "interior view of the grand iwan framing the courtyard",
    "35.jpg": "the mosque illuminated with golden artificial lights at night",
    "36.jpg": "close up of the intricate geometric marble floor patterns",
    "37.jpg": "tall vertical recesses housing the windows on the exterior wall",
    "38.jpg": "hanging lamp casting warm light onto the stone walls",
    "39.jpg": "wide angle view of the soaring entrance portal",
    "40.jpg": "the majestic skyline profile of the mosque's dome and minaret"
}

missing = [f for f in images if f not in CAPTIONS or not CAPTIONS[f].strip()]
if missing:
    print(f"WARNING: MISSING CAPTIONS FOR: {missing}")
else:
    print("All images have captions! You can proceed to Step 5.")


In [ ]:
# Step 5: Crop Images to 1024x1024 and Build metadata.jsonl
import json
TARGET_RES = 1024
os.makedirs("dataset", exist_ok=True)

def center_crop_resize(img, size):
    w, h = img.size
    scale = size / min(w, h)
    new_w, new_h = round(w * scale), round(h * scale)
    img = img.resize((new_w, new_h), Image.LANCZOS)
    left = (new_w - size) // 2
    top = (new_h - size) // 2
    return img.crop((left, top, left + size, top + size))

metadata_rows = []
for fname in images:
    base = os.path.splitext(fname)[0]
    img = Image.open(os.path.join(src_dir, fname)).convert("RGB")
    out_img = center_crop_resize(img, TARGET_RES)
    out_name = base + ".jpg"
    out_img.save(os.path.join("dataset", out_name), quality=95)

    raw_caption = CAPTIONS.get(fname, "")
    caption = f"sltnhsn, {raw_caption}"
    metadata_rows.append({"file_name": out_name, "text": caption})

with open(os.path.join("dataset", "metadata.jsonl"), "w", encoding="utf-8") as f:
    for row in metadata_rows:
        f.write(json.dumps(row) + "\n")
print("Images cropped and metadata.jsonl generated successfully!")


In [ ]:
# Step 6: Download Training Script & Configure
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/v0.31.0/examples/text_to_image/train_text_to_image_lora_sdxl.py
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="fp16")


In [ ]:
# Step 6.5: Record training settings (Required Deliverable!)
import json
TRAINING_SETTINGS = {
    "base_model": "stabilityai/stable-diffusion-xl-base-1.0",
    "vae": "madebyollin/sdxl-vae-fp16-fix",
    "method": "LoRA (rank 32) via diffusers train_text_to_image_lora_sdxl.py",
    "trigger_word": "sltnhsn",
    "resolution": 1024,
    "train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": "1e-4",
    "lr_scheduler": "constant",
    "max_train_steps": 1500,
    "checkpointing_steps": 100,  # Match this perfectly with the training script below!
    "seed": 42,
    "mixed_precision": "fp16",
    "rank": 32,
    "optimizer": "AdamW (bitsandbytes 8-bit Adam not usable on current Colab image)",
    "num_training_images": len(metadata_rows),
}

with open("training_settings.json", "w") as f:
    json.dump(TRAINING_SETTINGS, f, indent=2)

print("Saved to training_settings.json — include this file in your deliverables.")
print(json.dumps(TRAINING_SETTINGS, indent=2))


In [ ]:
# Step 7: Train! (Saves directly to Google Drive, auto-resumes)
xformers_flag = "--enable_xformers_memory_efficient_attention" if USE_XFORMERS else ""

!accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="dataset" \
  --caption_column="text" \
  --output_dir="/content/drive/MyDrive/sultan_hassan_sdxl_lora" \
  --mixed_precision="fp16" \
  --resolution=1024 \
  --center_crop \
  --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --rank=32 \
  --learning_rate=1e-4 \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --max_train_steps=1500 \
  --checkpointing_steps=100 \
  --resume_from_checkpoint="latest" \
  --seed=42 \
  --gradient_checkpointing \
  {xformers_flag}


In [ ]:
# Step 8: Generate Required Deliverable Test Images
from diffusers import DiffusionPipeline, AutoencoderKL
import torch, os, json

vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
).to("cuda")
pipe.load_lora_weights("/content/drive/MyDrive/sultan_hassan_sdxl_lora")

PROMPTS = [
    "sltnhsn, elevation",
    "sltnhsn, entrance gates",
    "sltnhsn, a courtyard with a central fountain at sunset",
    "sltnhsn, a mosque at night",
    "a modern glass office tower", 
]
SEEDS = [1000, 1001, 1002, 1003]

os.makedirs("test_images", exist_ok=True)
for p_idx, prompt in enumerate(PROMPTS):
    safe_name = prompt.replace(", ", "_").replace(" ", "-")[:40]
    for seed in SEEDS:
        generator = torch.Generator(device="cuda").manual_seed(seed)
        image = pipe(prompt, num_inference_steps=30, generator=generator).images[0]
        fname = f"test_images/{p_idx:02d}_{safe_name}_seed{seed}.png"
        image.save(fname)

!zip -r -q test_images.zip test_images
from google.colab import files
files.download('test_images.zip')


In [ ]:
# Step 9: Download the final deliverables
from google.colab import files
files.download('training_settings.json')
print("Downloaded training_settings.json!")
